# 02 — Image embeddings with frozen foundation models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martinamorales/skin-lesion-triage/blob/main/notebooks/02_embeddings.ipynb)

Instead of training a neural network from scratch, we use two **foundation models**
that were already trained on millions of images, and keep them **frozen** (unchanged):

| Model | Trained on | Embedding size |
|---|---|---|
| **DINOv2** (Meta, `dinov2-base`) | 142M general natural images | 768 |
| **BiomedCLIP** (Microsoft) | 15M figure–caption pairs from biomedical papers | 512 |

Each model turns every HAM10000 image into an **embedding**: a short vector of numbers
that summarises what the image looks like. A frozen model always gives the same embedding
for the same image, so we compute them **once** here (on a GPU) and save them. Every later
step (classifier, calibration, evaluation) just loads these files and runs on a CPU.

**Run this notebook on Colab with a GPU:** *Runtime → Change runtime type → T4 GPU*.

## Setup

Clones the repository and installs the package together with the `embeddings` dependency
group (PyTorch, Hugging Face `transformers`, `open_clip`). `BRANCH` selects which version
of the code to use; leave it as `main` unless you are testing a pull request.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/martinamorales/skin-lesion-triage.git"
BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and Path.cwd().name != "skin-lesion-triage":
    if not Path("skin-lesion-triage").exists():
        !git clone -q --branch {BRANCH} {REPO_URL}
    os.chdir("skin-lesion-triage")
    # Dependency groups (PEP 735) need a recent pip.
    !pip install -q --upgrade pip
    !pip install -q . --group embeddings --group dev
elif Path.cwd().name == "notebooks":
    os.chdir("..")

print("Running on Colab:", IN_COLAB)

Check that a GPU is available. On a CPU everything still works, but computing the
embeddings would take hours instead of minutes.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. On Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.")

## Run the tests, including the real models

Locally and in CI, the tests that load the real models are **skipped** because PyTorch is
not installed. Here it is, so they run for real: each model must return one finite row per
image with the expected size, and the same output twice for the same input.

In [ ]:
!{sys.executable} -m pytest tests/test_encoders.py tests/test_embeddings.py -v

## Data

Download HAM10000 (skipped if it is already there) and load the metadata. We embed the
images in the order of the metadata table; the image ids are saved next to the embeddings,
so later steps join by `image_id` and never rely on row order.

In [ ]:
!{sys.executable} -m skin_lesion_triage.download

In [ ]:
import pandas as pd

DATA_DIR = Path("data/ham10000")
IMAGES_DIR = DATA_DIR / "images"

meta = pd.read_csv(DATA_DIR / "HAM10000_metadata.csv")
image_ids = meta["image_id"].tolist()
print(f"{len(image_ids):,} images to embed")

## Where to save the embeddings

On Colab everything in `/content` is **deleted when the session ends**, so we save the
embeddings to Google Drive (Colab will ask for permission). Each file is a few tens of MB.
Locally they go to `data/embeddings/`, which is git-ignored.

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/skin-lesion-triage/embeddings")
else:
    OUT_DIR = Path("data/embeddings")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saving embeddings to", OUT_DIR)

## Smoke test: 10 images first

Before spending GPU time on 10,015 images, try 10 with each model. This catches problems
in seconds instead of after a long run, and the timing gives an estimate for the full run.

In [ ]:
import time

import numpy as np

from skin_lesion_triage.embeddings import compute_embeddings
from skin_lesion_triage.encoders import ENCODERS, get_encoder

for name in ENCODERS:
    encoder = get_encoder(name)
    start = time.perf_counter()
    sample = compute_embeddings(image_ids[:10], IMAGES_DIR, encoder, batch_size=10)
    seconds = time.perf_counter() - start
    print(
        f"{name:>10}: shape {sample.shape}, all finite: {np.isfinite(sample).all()}, "
        f"{seconds:.1f} s for 10 images"
    )
    del encoder
    torch.cuda.empty_cache()

The first call of each model includes some one-time warm-up, so the real speed is better
than this timing suggests.

## Full run

For each model: load it once, embed all images, save the result, then free the GPU memory
before loading the next model.

`compute_embeddings` is called on chunks of 1,000 images only to print progress. This does
not change the result: the tests check that embeddings do not depend on how images are
grouped. If a file already exists the model is skipped, so if Colab disconnects you can
re-run this cell and it continues where it stopped.

In [ ]:
from skin_lesion_triage.embeddings import save_embeddings

BATCH_SIZE = 64
CHUNK = 1_000

for name in ENCODERS:
    path = OUT_DIR / f"{name}.npz"
    if path.exists():
        print(f"[skip] {path.name} already exists")
        continue

    encoder = get_encoder(name)
    start = time.perf_counter()
    chunks = []
    for i in range(0, len(image_ids), CHUNK):
        chunk_ids = image_ids[i : i + CHUNK]
        chunks.append(compute_embeddings(chunk_ids, IMAGES_DIR, encoder, batch_size=BATCH_SIZE))
        done = i + len(chunk_ids)
        print(f"\r{name}: {done:,}/{len(image_ids):,} images", end="", flush=True)

    embeddings = np.concatenate(chunks)
    save_embeddings(path, image_ids, embeddings)
    minutes = (time.perf_counter() - start) / 60
    print(f"\n[saved] {path.name}: {embeddings.shape} in {minutes:.1f} min")

    del encoder
    torch.cuda.empty_cache()

## Check the saved files

Reload each file from Drive and check it: right number of rows, ids in the same order as
the metadata, no missing or infinite values.

In [ ]:
from skin_lesion_triage.embeddings import load_embeddings

saved = {}
for name in ENCODERS:
    ids, emb = load_embeddings(OUT_DIR / f"{name}.npz")
    assert ids == image_ids, f"{name}: image ids do not match the metadata"
    assert np.isfinite(emb).all(), f"{name}: contains NaN or infinite values"
    saved[name] = emb
    print(f"{name:>10}: {emb.shape[0]:,} images x {emb.shape[1]} numbers, OK")

## Sanity check: do the embeddings mean something?

The files have the right shape, but are the numbers *useful*? A cheap check, before any
classifier: images of the **same lesion** (same mole, photographed more than once) should
look more alike to the model than two **random** images.

We measure "alike" with **cosine similarity**: 1 means the two vectors point in exactly
the same direction, 0 means they are unrelated.

In [ ]:
rng = np.random.default_rng(0)

# Pairs of different images of the same lesion.
same_pairs = []
for _, group in meta.groupby("lesion_id"):
    idx = group.index.to_numpy()
    if len(idx) >= 2:
        same_pairs.append(idx[:2])
same_pairs = np.array(same_pairs)

# The same number of random pairs of images.
random_pairs = rng.integers(0, len(meta), size=same_pairs.shape)


def cosine(emb: np.ndarray, pairs: np.ndarray) -> np.ndarray:
    a, b = emb[pairs[:, 0]], emb[pairs[:, 1]]
    return (a * b).sum(axis=1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))


rows = []
for name, emb in saved.items():
    rows.append({
        "model": name,
        "same lesion (mean)": cosine(emb, same_pairs).mean(),
        "random pair (mean)": cosine(emb, random_pairs).mean(),
    })
pd.DataFrame(rows).set_index("model").round(3)

If the models are doing their job, the *same lesion* column is clearly higher than the
*random pair* column for both models. Note this is **not** a diagnosis result yet: it only
shows that the embeddings capture what an image looks like.

## Next

Notebook 03 trains a simple classifier (logistic regression) on these embeddings, using
the lesion-level train/val/test splits, and compares DINOv2 with BiomedCLIP.